In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("/content/lenta-ru-news.csv")  # Укажите свой путь

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Файл не найден: {DATA_PATH}")

print("Имя файла:", DATA_PATH.name)
print("Размер, байт:", DATA_PATH.stat().st_size)
print("Размер, ГиБ:", round(DATA_PATH.stat().st_size / 1024**3, 3))

sample = pd.read_csv(
    DATA_PATH,
    nrows=5,
    dtype=str,
    encoding="utf-8",
)

print("\nСтолбцы:", sample.columns.tolist())
print("Прочитано записей:", len(sample))

print("\nПервые записи:")
for i, row in sample.iterrows():
    print(f"\n--- Запись {i + 1} ---")
    for column, value in row.items():
        preview = (
            "<пропуск>"
            if pd.isna(value)
            else str(value).replace("\n", " ")[:350]
        )
        print(f"{column}: {preview}")

Имя файла: lenta-ru-news.csv
Размер, байт: 2084746431
Размер, ГиБ: 1.942

Столбцы: ['url', 'title', 'text', 'topic', 'tags', 'date']
Прочитано записей: 5

Первые записи:

--- Запись 1 ---
url: https://lenta.ru/news/1914/09/16/hungarnn/
title: 1914. Русские войска вступили в пределы Венгрии  
text: Бои у Сопоцкина и Друскеник закончились отступлением германцев. Неприятель, приблизившись с севера к Осовцу начал артиллерийскую борьбу с крепостью. В артиллерийском бою принимают участие тяжелые калибры. С раннего утра 14 сентября огонь достиг значительного напряжения. Попытка германской пехоты пробиться ближе к крепости отражена. В Галиции мы зан
topic: Библиотека
tags: Первая мировая
date: 1914/09/16

--- Запись 2 ---
url: https://lenta.ru/news/1914/09/16/lermontov/
title: 1914. Празднование столетия М.Ю. Лермонтова отложено 
text: Министерство народного просвещения, в виду происходящих чрезвычайных событий, признало соответственным в день годовщины со дня рождения М.Ю. Лермонтова (2-го ок

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd

DATA_PATH = Path("/content/lenta-ru-news.csv")

columns = ["url", "title", "text", "topic", "tags", "date"]

total_rows = 0
empty_counts = Counter({column: 0 for column in columns})
topic_counts = Counter()
tag_counts = Counter()

date_min = None
date_max = None
invalid_nonempty_dates = 0
dates_before_1999 = 0

nonempty_text_count = 0
text_length_sum = 0
text_length_min = None
text_length_max = None

reader = pd.read_csv(
    DATA_PATH,
    dtype=str,
    encoding="utf-8",
    keep_default_na=False,
    chunksize=20_000,
    on_bad_lines="error",
)

for chunk_number, chunk in enumerate(reader, start=1):
    total_rows += len(chunk)

    for column in columns:
        values = chunk[column].str.strip()
        empty_counts[column] += int(values.eq("").sum())

    topics = chunk["topic"].str.strip().replace("", "<пусто>")
    tags = chunk["tags"].str.strip().replace("", "<пусто>")

    topic_counts.update(topics.value_counts().to_dict())
    tag_counts.update(tags.value_counts().to_dict())

    raw_dates = chunk["date"].str.strip()
    dates = pd.to_datetime(
        raw_dates,
        format="%Y/%m/%d",
        errors="coerce",
    )

    invalid_nonempty_dates += int(
        (raw_dates.ne("") & dates.isna()).sum()
    )
    dates_before_1999 += int(
        (dates < pd.Timestamp("1999-01-01")).sum()
    )

    valid_dates = dates.dropna()

    if not valid_dates.empty:
        current_min = valid_dates.min()
        current_max = valid_dates.max()

        date_min = (
            current_min if date_min is None
            else min(date_min, current_min)
        )
        date_max = (
            current_max if date_max is None
            else max(date_max, current_max)
        )

    lengths = chunk["text"].str.strip().str.len()
    nonempty_lengths = lengths[lengths > 0]

    if not nonempty_lengths.empty:
        nonempty_text_count += len(nonempty_lengths)
        text_length_sum += int(nonempty_lengths.sum())

        current_min = int(nonempty_lengths.min())
        current_max = int(nonempty_lengths.max())

        text_length_min = (
            current_min if text_length_min is None
            else min(text_length_min, current_min)
        )
        text_length_max = (
            current_max if text_length_max is None
            else max(text_length_max, current_max)
        )

    if chunk_number % 10 == 0:
        print(f"Обработано записей: {total_rows:,}")

print("\n=== ИТОГИ ===")
print("Всего записей:", total_rows)
print("Размер файла, байт:", DATA_PATH.stat().st_size)

print("\n=== ПУСТЫЕ ЗНАЧЕНИЯ ===")
for column in columns:
    count = empty_counts[column]
    percent = count / total_rows * 100 if total_rows else 0
    print(f"{column}: {count:,} ({percent:.2f}%)")

print("\n=== РУБРИКИ: ВСЕ ===")
for topic, count in topic_counts.most_common():
    print(f"{topic}: {count:,}")

print("\n=== ТЕГИ: 40 САМЫХ ЧАСТЫХ ===")
print(
    "Количество уникальных непустых тегов:",
    len([tag for tag in tag_counts if tag != "<пусто>"])
)
for tag, count in tag_counts.most_common(40):
    print(f"{tag}: {count:,}")

print("\n=== ДАТЫ ===")
print(
    "Минимальная распознанная дата:",
    date_min.date() if date_min is not None else "нет"
)
print(
    "Максимальная распознанная дата:",
    date_max.date() if date_max is not None else "нет"
)
print("Непустые даты, не распознанные как YYYY/MM/DD:",
      invalid_nonempty_dates)
print("Записи с датой до 1999 года:", dates_before_1999)

print("\n=== ДЛИНА НЕПУСТОГО ТЕКСТА ===")
print("Количество непустых текстов:", nonempty_text_count)
print("Минимальная длина, символов:", text_length_min)
print("Максимальная длина, символов:", text_length_max)
print(
    "Средняя длина, символов:",
    round(text_length_sum / nonempty_text_count, 1)
    if nonempty_text_count else "нет"
)

Обработано записей: 200,000
Обработано записей: 400,000
Обработано записей: 600,000
Обработано записей: 800,000

=== ИТОГИ ===
Всего записей: 800975
Размер файла, байт: 2084746431

=== ПУСТЫЕ ЗНАЧЕНИЯ ===
url: 0 (0.00%)
title: 0 (0.00%)
text: 5 (0.00%)
topic: 62,002 (7.74%)
tags: 27,219 (3.40%)
date: 0 (0.00%)

=== РУБРИКИ: ВСЕ ===
Россия: 160,445
Мир: 136,621
Экономика: 79,528
Спорт: 64,413
<пусто>: 62,002
Культура: 53,797
Бывший СССР: 53,402
Наука и техника: 53,136
Интернет и СМИ: 44,663
Из жизни: 27,605
Дом: 21,734
Силовые структуры: 19,596
Ценности: 7,766
Бизнес: 7,399
Путешествия: 6,408
69-я параллель: 1,268
Крым: 666
Культпросвет: 340
Легпром: 114
Библиотека: 65
Оружие: 3
ЧМ-2014: 2
Сочи: 1
МедНовости: 1

=== ТЕГИ: 40 САМЫХ ЧАСТЫХ ===
Количество уникальных непустых тегов: 94
Все: 453,762
Политика: 40,716
Общество: 35,202
<пусто>: 27,219
Украина: 22,523
Происшествия: 19,825
Госэкономика: 15,476
Футбол: 15,239
Кино: 10,720
Интернет: 9,319
Бизнес: 9,146
Следствие и суд: 9,133
Наука:

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

DATA_PATH = Path("/content/lenta-ru-news.csv")

tag_totals = Counter()
tag_topics = defaultdict(Counter)

reader = pd.read_csv(
    DATA_PATH,
    usecols=["topic", "tags"],
    dtype=str,
    encoding="utf-8",
    keep_default_na=False,
    chunksize=20_000,
    on_bad_lines="error",
)

for chunk in reader:
    chunk["topic"] = (
        chunk["topic"].str.strip().replace("", "<пусто>")
    )
    chunk["tags"] = (
        chunk["tags"].str.strip().replace("", "<пусто>")
    )

    pairs = chunk.groupby(["tags", "topic"]).size()

    for (tag, topic), count in pairs.items():
        count = int(count)
        tag_totals[tag] += count
        tag_topics[tag][topic] += count

print("=== ВСЕ ТЕГИ И СВЯЗАННЫЕ РУБРИКИ ===")

for tag in sorted(tag_totals):
    top_topics = "; ".join(
        f"{topic}: {count:,}"
        for topic, count in tag_topics[tag].most_common(3)
    )

    print(
        f"{tag!r} | всего: {tag_totals[tag]:,} "
        f"| рубрики: {top_topics}"
    )

=== ВСЕ ТЕГИ И СВЯЗАННЫЕ РУБРИКИ ===
'69-я параллель' | всего: 1 | рубрики: <пусто>: 1
'<пусто>' | всего: 27,219 | рубрики: Силовые структуры: 8,373; Спорт: 6,519; Россия: 5,438
'Coцсети' | всего: 3,635 | рубрики: Интернет и СМИ: 2,924; <пусто>: 711
'Авто' | всего: 204 | рубрики: Спорт: 141; <пусто>: 63
'Автобизнес' | всего: 310 | рубрики: Бизнес: 310
'Аналитика рынка' | всего: 28 | рубрики: Легпром: 28
'Английский футбол' | всего: 348 | рубрики: <пусто>: 348
'Белоруссия' | всего: 1,344 | рубрики: Бывший СССР: 916; <пусто>: 428
'Бизнес' | всего: 9,146 | рубрики: Экономика: 4,264; Бизнес: 3,960; <пусто>: 922
'Бокс и ММА' | всего: 3,549 | рубрики: Спорт: 2,443; <пусто>: 1,106
'Вещи' | всего: 479 | рубрики: Из жизни: 424; <пусто>: 55
'Вирусные ролики' | всего: 514 | рубрики: Интернет и СМИ: 474; <пусто>: 40
'Вкусы' | всего: 93 | рубрики: Ценности: 54; <пусто>: 39
'Внешний вид' | всего: 1,047 | рубрики: Ценности: 727; <пусто>: 320
'Вооружение' | всего: 3 | рубрики: Оружие: 3
'Все' | всего:

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import random
import re
import pandas as pd

DATA_PATH = Path("/content/lenta-ru-news.csv")

selected_tags = [
    "Кино", "Музыка", "Театр",
    "Гаджеты", "Софт", "Техника",
]

education_pattern = re.compile(
    r"\b(?:"
    r"образовани[еяю]"
    r"|обучени[еяю]"
    r"|учебн\w*"
    r"|онлайн[-\s]+курс\w*"
    r"|курс\w*\s+(?:по|для)\b"
    r"|лекци\w*"
    r"|репетитор\w*"
    r"|егэ"
    r"|огэ"
    r"|стипенди\w*"
    r"|студент\w*"
    r"|университет\w*"
    r"|школ\w*"
    r")\b",
    flags=re.IGNORECASE,
)

rng = random.Random(42)
seen = Counter()
examples = defaultdict(list)

def keep_example(group, record, limit):
    seen[group] += 1
    sample = examples[group]

    if len(sample) < limit:
        sample.append(record)
    else:
        index = rng.randrange(seen[group])
        if index < limit:
            sample[index] = record

reader = pd.read_csv(
    DATA_PATH,
    dtype=str,
    encoding="utf-8",
    keep_default_na=False,
    chunksize=20_000,
    on_bad_lines="error",
)

for chunk in reader:
    tags = chunk["tags"].str.strip()
    titles = chunk["title"].str.strip()
    has_text = chunk["text"].str.strip().ne("")

    tag_mask = tags.isin(selected_tags) & has_text
    education_mask = titles.str.contains(education_pattern) & has_text

    candidates = chunk.loc[tag_mask | education_mask]

    for record in candidates.to_dict("records"):
        tag = record["tags"].strip()

        if tag in selected_tags:
            keep_example(tag, record, limit=3)

        if education_pattern.search(record["title"].strip()):
            keep_example(
                "Образование — поиск по заголовку",
                record,
                limit=10,
            )

print("=== КАНДИДАТЫ: КОЛИЧЕСТВО ===")
groups = selected_tags + ["Образование — поиск по заголовку"]

for group in groups:
    print(f"{group}: {seen[group]:,}")

print("\n=== ПРИМЕРЫ ДЛЯ ПРОВЕРКИ ===")

for group in groups:
    print(f"\n### {group}")

    for number, record in enumerate(examples[group], start=1):
        text = " ".join(record["text"].split())

        print(f"\nПример {number}")
        print("Заголовок:", record["title"])
        print("Рубрика:", record["topic"] or "<пусто>")
        print("Тег:", record["tags"] or "<пусто>")
        print("Дата:", record["date"])
        print("Текст:", text[:1200])

        if len(text) > 1200:
            print("[Текст сокращён для вывода]")

=== КАНДИДАТЫ: КОЛИЧЕСТВО ===
Кино: 10,720
Музыка: 7,405
Театр: 1,393
Гаджеты: 2,351
Софт: 663
Техника: 1,355
Образование — поиск по заголовку: 10,655

=== ПРИМЕРЫ ДЛЯ ПРОВЕРКИ ===

### Кино

Пример 1
Заголовок: Звезду «12 лет рабства» позвали сыграть злодея в «Докторе Стрэндже»
Рубрика: Культура
Тег: Кино
Дата: 2015/01/23
Текст: Актер Чиветел Эджиофор, исполнивший главную роль в драме «12 лет рабства», приглашен сняться в картине «Доктор Стрэндж» по одноименной серии комиксов Marvel, сообщает The Hollywood Reporter. Как уточняет издание, актер в настоящее время находится на начальной стадии переговоров со студией Marvel. Предполагается, что Эджиофор сыграет злодея. Лента выйдет в прокат в ноябре 2016 года. Главную роль в ней исполнит Бенедикт Камбербэтч. Режиссером картины выступит Скотт Дерриксон, известный по фильмам ужасов «Восставший из ада 5: Инферно» и «Шесть демонов Эмили Роуз». Подробности сюжета и полный актерский состав «Доктора Стрэнджа» пока не уточняются. Чиветел Эджиофор

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import re
import pandas as pd

DATA_PATH = Path("/content/lenta-ru-news.csv")

html_pattern = re.compile(
    r"</?[A-Za-z][A-Za-z0-9]*(?:\s[^<>]*?)?\s*/?>"
)

counts = Counter()
seen_hashes = set()
examples = {}

def save_example(kind, row_id, title, text):
    if kind not in examples:
        examples[kind] = {
            "row_id": row_id,
            "title": title,
            "text": text[:700],
        }

reader = pd.read_csv(
    DATA_PATH,
    usecols=["title", "text"],
    dtype=str,
    encoding="utf-8",
    keep_default_na=False,
    chunksize=20_000,
    on_bad_lines="error",
)

total = 0

for chunk in reader:
    for title, raw_text in zip(chunk["title"], chunk["text"]):
        row_id = total
        total += 1

        text = " ".join(raw_text.split())

        empty = not text
        short = bool(text) and len(text) < 100
        possible_html = bool(html_pattern.search(raw_text))
        replacement_char = "\ufffd" in raw_text
        duplicate = False

        if text:
            digest = hashlib.sha256(
                text.encode("utf-8")
            ).digest()

            duplicate = digest in seen_hashes
            seen_hashes.add(digest)

        flags = {
            "Пустой текст": empty,
            "Непустой текст короче 100 символов": short,
            "Возможные HTML-теги": possible_html,
            "Символ замены �": replacement_char,
            "Повтор текста сверх первого экземпляра": duplicate,
        }

        for kind, flag in flags.items():
            if flag:
                counts[kind] += 1
                save_example(kind, row_id, title, raw_text)

        # Дубликаты учитываются отдельно от дефектов содержимого.
        review_flag = empty or short or possible_html or replacement_char
        counts["Хотя бы один признак для проверки"] += int(review_flag)

    if total % 100_000 == 0:
        print(f"Обработано: {total:,}")

print("\n=== ТЕХНИЧЕСКОЕ КАЧЕСТВО ===")
print("Всего записей:", total)

metrics = [
    "Пустой текст",
    "Непустой текст короче 100 символов",
    "Возможные HTML-теги",
    "Символ замены �",
    "Повтор текста сверх первого экземпляра",
    "Хотя бы один признак для проверки",
]

for kind in metrics:
    count = counts[kind]
    percent = count / total * 100 if total else 0
    print(f"{kind}: {count:,} ({percent:.5f}%)")

without_flags = total - counts["Хотя бы один признак для проверки"]
percent = without_flags / total * 100 if total else 0

print(
    f"Без перечисленных признаков: "
    f"{without_flags:,} ({percent:.5f}%)"
)
print("Уникальных непустых нормализованных текстов:", len(seen_hashes))

print("\n=== ПРИМЕРЫ ДЛЯ ПРОВЕРКИ ===")
for kind, example in examples.items():
    print(f"\n{kind}")
    print("Номер записи, начиная с 0:", example["row_id"])
    print("Заголовок:", example["title"])
    print("Фрагмент:", repr(example["text"]))


Обработано: 100,000
Обработано: 200,000
Обработано: 300,000
Обработано: 400,000
Обработано: 500,000
Обработано: 600,000
Обработано: 700,000
Обработано: 800,000

=== ТЕХНИЧЕСКОЕ КАЧЕСТВО ===
Всего записей: 800975
Пустой текст: 5 (0.00062%)
Непустой текст короче 100 символов: 1,483 (0.18515%)
Возможные HTML-теги: 7 (0.00087%)
Символ замены �: 0 (0.00000%)
Повтор текста сверх первого экземпляра: 937 (0.11698%)
Хотя бы один признак для проверки: 1,495 (0.18665%)
Без перечисленных признаков: 799,480 (99.81335%)
Уникальных непустых нормализованных текстов: 800033

=== ПРИМЕРЫ ДЛЯ ПРОВЕРКИ ===

Непустой текст короче 100 символов
Номер записи, начиная с 0: 1775
Заголовок: В самолетах Китая и Южной Кореи наступит 2000 год
Фрагмент: 'Южная Корея и Китай намерены организова'

Повтор текста сверх первого экземпляра
Номер записи, начиная с 0: 33179
Заголовок: Европейский союз составит досье на антиглобалистов
Фрагмент: 'Лидеры стран-участниц Европейского Союза приняли решение о координировании дейс

In [ ]:
from pathlib import Path
import hashlib
import random
import re
import pandas as pd

DATA_PATH = Path("/content/lenta-ru-news.csv")
AUDIT_PATH = Path("/content/manual_quality_audit.csv")

SEED = 42
NORMAL_SAMPLE_SIZE = 100
FLAGGED_SAMPLE_SIZE = 20

html_pattern = re.compile(
    r"</?[A-Za-z][A-Za-z0-9]*(?:\s[^<>]*?)?\s*/?>"
)

rng = random.Random(SEED)

normal_reservoir = []
flagged_reservoir = []
seen_hashes = set()

def reservoir_add(reservoir, item, seen_count, limit):
    if len(reservoir) < limit:
        reservoir.append(item)
    else:
        index = rng.randrange(seen_count)
        if index < limit:
            reservoir[index] = item

reader = pd.read_csv(
    DATA_PATH,
    usecols=["url", "title", "text", "topic", "tags", "date"],
    dtype=str,
    encoding="utf-8",
    keep_default_na=False,
    chunksize=20_000,
    on_bad_lines="error",
)

normal_seen = 0
flagged_seen = 0
row_id = 0

for chunk in reader:
    for record in chunk.to_dict("records"):
        raw_text = record["text"]
        normalized_text = " ".join(raw_text.split())

        empty = not normalized_text
        short = bool(normalized_text) and len(normalized_text) < 100
        possible_html = bool(html_pattern.search(raw_text))
        replacement_char = "\ufffd" in raw_text

        duplicate = False
        if normalized_text:
            digest = hashlib.sha256(
                normalized_text.encode("utf-8")
            ).digest()
            duplicate = digest in seen_hashes
            seen_hashes.add(digest)

        flags = []
        if empty:
            flags.append("empty_text")
        if short:
            flags.append("short_under_100")
        if possible_html:
            flags.append("possible_html")
        if replacement_char:
            flags.append("replacement_char")
        if duplicate:
            flags.append("duplicate")

        audit_record = {
            "source_row_id": row_id,
            "url": record["url"],
            "title": record["title"],
            "text": raw_text,
            "topic": record["topic"],
            "tags": record["tags"],
            "date": record["date"],
            "automatic_flags": "|".join(flags) if flags else "none",
            "technical_readable": "",
            "content_complete": "",
            "content_relevant": "",
            "manual_noise": "",
            "comment": "",
        }

        if flags:
            flagged_seen += 1
            reservoir_add(
                flagged_reservoir,
                audit_record,
                flagged_seen,
                FLAGGED_SAMPLE_SIZE,
            )
        else:
            normal_seen += 1
            reservoir_add(
                normal_reservoir,
                audit_record,
                normal_seen,
                NORMAL_SAMPLE_SIZE,
            )

        row_id += 1

audit_df = pd.DataFrame(
    normal_reservoir + flagged_reservoir
)

audit_df.to_csv(
    AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Файл создан:", AUDIT_PATH.name)
print("Всего строк для ручного аудита:", len(audit_df))
print("\nРаспределение автоматических признаков:")
print(audit_df["automatic_flags"].value_counts())

print("\nПервые 10 строк:")
display(
    audit_df[
        [
            "source_row_id",
            "title",
            "topic",
            "tags",
            "automatic_flags",
            "technical_readable",
            "content_complete",
            "content_relevant",
            "manual_noise",
            "comment",
        ]
    ].head(10)
)

Файл создан: manual_quality_audit.csv
Всего строк для ручного аудита: 120

Распределение автоматических признаков:
automatic_flags
none                         100
short_under_100|duplicate     12
short_under_100                8
Name: count, dtype: int64

Первые 10 строк:


,source_row_id,title,topic,tags,automatic_flags,technical_readable,content_complete,content_relevant,manual_noise,comment
0,627060,МВФ рассказал России о правильном расходовании...,Экономика,Госэкономика,none,,,,,
1,453754,Цирк Никулина пригрозил отъездом из России,Культура,Театр,none,,,,,
2,29749,"Водолазы отработают детали подъема ""Курска"" на...",Россия,Все,none,,,,,
3,233163,Аэропорту Сухуми отказали в международном приз...,Бывший СССР,Все,none,,,,,
4,799087,Супруги прожили вместе 68 лет и умерли с разни...,,Люди,none,,,,,
5,552919,Восемь человек на юге Киргизии госпитализирова...,Бывший СССР,Средняя Азия,none,,,,,
6,376106,Собянин попросил уравнять в правах все такси в...,Экономика,Все,none,,,,,
7,735062,На Зидана появился очередной претендент,Спорт,Футбол,none,,,,,
8,299178,Украинские националисты потребовали ограничить...,Бывший СССР,Все,none,,,,,
9,153025,"Михаэль Шумахер озвучил мультфильм ""Тачки""",Спорт,Все,none,,,,,


In [ ]:
import pandas as pd
from pathlib import Path
from openpyxl.styles import Alignment
from google.colab import files

CSV_PATH = Path("/content/manual_quality_audit.csv")
XLSX_PATH = Path("/content/manual_quality_audit.xlsx")

audit = pd.read_csv(
    CSV_PATH,
    dtype=str,
    encoding="utf-8-sig",
    keep_default_na=False,
)

print("Количество записей:", len(audit))
print("Количество столбцов:", len(audit.columns))

assert len(audit) == 120, "Ожидалось 120 записей — пришли вывод проверки."
assert len(audit.columns) == 13, "Ожидалось 13 столбцов."

# Не допускаем незаметного сокращения длинных текстов при экспорте.
assert not audit["text"].str.len().gt(32767).any(), (
    "Есть текст длиннее ограничения ячейки Excel. "
    "Пришли эту ошибку — изменим формат проверки."
)

with pd.ExcelWriter(XLSX_PATH, engine="openpyxl") as writer:
    audit.to_excel(writer, index=False, sheet_name="Аудит")

    ws = writer.sheets["Аудит"]
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    widths = {
        "A": 16, "B": 25, "C": 55, "D": 100,
        "E": 22, "F": 22, "G": 15, "H": 30,
        "I": 22, "J": 22, "K": 22, "L": 18, "M": 45,
    }

    for column, width in widths.items():
        ws.column_dimensions[column].width = width

    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True,
            )
            if isinstance(cell.value, str):
                cell.data_type = "s"

    for row_number in range(2, ws.max_row + 1):
        ws.row_dimensions[row_number].height = 120

assert XLSX_PATH.is_file() and XLSX_PATH.stat().st_size > 0
files.download(str(XLSX_PATH))

Количество записей: 120
Количество столбцов: 13


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Предобработка

In [ ]:
!pip -q install pymorphy3 pymorphy3-dicts-ru

from pathlib import Path
import re
import multiprocessing
import numpy as np
import pandas as pd
import pymorphy3
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed

DATA_PATH = Path("/content/lenta-ru-news.csv") #Надеюсь ты не забудешь сменить путь к файлу)
RANDOM_STATE = 42

MAX_CHARS     = 1200
MIN_TEXT_LEN  = 60
MAX_PER_CLASS = 30_000

# Тут можно поменять маппинг, если изменить в "Интернет и СМИ" "Технологии" на "Новости",
# то небольшая часть ошибок пропадет, я пробовал и так и так разница минимальная, поэтому оставил как есть
# Если захочешь можешь поменять
TOPIC_MAP = {
    "Культура":          "Развлечения",
    "Ценности":          "Развлечения",
    "Культпросвет":      "Развлечения",
    "Спорт":             "Спорт",
    "Наука и техника":   "Технологии",
    "Интернет и СМИ":    "Технологии", # <- Вот тут "Новости"
    "Россия":            "Новости",
    "Мир":               "Новости",
    "Бывший СССР":       "Новости",
    "Экономика":         "Новости",
    "Бизнес":            "Новости",
    "Силовые структуры": "Новости",
    "Из жизни":          "Новости",
    "Дом":               "Новости",
    "Путешествия":       "Новости",
    "69-я параллель":    "Новости",
    "Крым":              "Новости",
}
TOPICS_KEEP = set(TOPIC_MAP.keys())

_re_html  = re.compile(r"<[^>]+>")
_re_url   = re.compile(r"https?://\S+|www\.\S+")
_re_mail  = re.compile(r"\S+@\S+")
_re_nonal = re.compile(r"[^а-яa-z\s]")
_re_space = re.compile(r"\s+")

_MORPH = pymorphy3.MorphAnalyzer()
_LEMMA_CACHE = {}


def _lemma(token: str) -> str:
    cached = _LEMMA_CACHE.get(token)
    if cached is not None:
        return cached
    lemma = _MORPH.parse(token)[0].normal_form
    _LEMMA_CACHE[token] = lemma
    return lemma


def clean_and_lemmatize(value: str) -> str:
    text = "" if value is None else str(value)
    if len(text) > MAX_CHARS:
        text = text[:MAX_CHARS]
    text = _re_html.sub(" ", text)
    text = _re_url.sub(" ", text)
    text = _re_mail.sub(" ", text)
    text = text.lower().replace("ё", "е")
    text = _re_nonal.sub(" ", text)
    text = _re_space.sub(" ", text).strip()
    if not text:
        return ""
    tokens = [_lemma(t) for t in text.split()]
    return " ".join(t for t in tokens if len(t) > 1)


# Загрузка
parts = []
for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["title", "text", "topic"],
    dtype=str,
    encoding="utf-8",
    keep_default_na=False,
    chunksize=100_000,
    on_bad_lines="error",
):
    chunk["topic"] = chunk["topic"].str.strip()
    chunk = chunk[chunk["topic"].isin(TOPICS_KEEP)]
    if not chunk.empty:
        parts.append(chunk.copy())

df = pd.concat(parts, ignore_index=True)
print("Загружено записей нужных рубрик:", len(df))

before = len(df)
df = df.drop_duplicates(subset=["title", "text"]).reset_index(drop=True)
print(f"Удалено точных дублей (raw): {before - len(df)}")

# Маппинг
df["category"] = df["topic"].map(TOPIC_MAP)
df = df[df["category"].notna()].copy()
print("\nРаспределение по 4 темам:")
print(df["category"].value_counts())

# Балансировка
df = (
    df.sample(frac=1, random_state=RANDOM_STATE)
      .groupby("category", group_keys=False, sort=False)
      .head(MAX_PER_CLASS)
      .reset_index(drop=True)
)
print("\nПосле балансировки:")
print(df["category"].value_counts())

df["raw_full"] = (
    df["title"].fillna("") + ". " + df["text"].fillna("")
).str.slice(0, MAX_CHARS + 200)

# Лемматизация
N_JOBS = min(4, max(1, multiprocessing.cpu_count() - 1))
print("\nВоркеров:", N_JOBS)

cleaned = Parallel(
    n_jobs=N_JOBS, backend="threading", batch_size=500, verbose=5,
)(
    delayed(clean_and_lemmatize)(t) for t in df["raw_full"].tolist()
)
df["text_clean"] = cleaned

df = df[df["text_clean"].str.len() >= MIN_TEXT_LEN].copy()

before = len(df)
df = df.drop_duplicates(subset=["text_clean"]).reset_index(drop=True)
print(f"\nУдалено дубликатов по text_clean: {before - len(df)}")
print("Осталось записей:", len(df))

X = df["text_clean"].to_numpy()
y = df["category"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=RANDOM_STATE, stratify=y_train,
)
print("\nTrain / Val / Test:", X_tr.shape, X_val.shape, X_test.shape)

Векторизация TF-IDF и подбор гиперпараметра C на валидации

In [ ]:
import gc
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Векторизация
features = FeatureUnion([
    ("word", TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2),
        min_df=3, max_df=0.9, sublinear_tf=True,
        max_features=200_000, dtype=np.float32,
    )),
    ("char", TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5),
        min_df=5, max_df=0.9, sublinear_tf=True,
        max_features=100_000, dtype=np.float32,
    )),
])

print("Считаю TF-IDF на train...")
X_tr_vec = features.fit_transform(X_tr)
print("Train:", X_tr_vec.shape, f"{X_tr_vec.data.nbytes/1e6:.0f} MB")

print("Считаю TF-IDF на val...")
X_val_vec = features.transform(X_val)
print("Val:  ", X_val_vec.shape)

print("Считаю TF-IDF на test...")
X_test_vec = features.transform(X_test)
print("Test: ", X_test_vec.shape)
gc.collect()


# Подбор C для LinearSVC на val
print("\nПодбор C на val для LinearSVC:")
svc_scores = {}
for C in [0.03, 0.1, 0.3, 1.0, 3.0]:
    clf = LinearSVC(
        C=C, class_weight="balanced", dual="auto",
        random_state=RANDOM_STATE, max_iter=15_000,
    )
    clf.fit(X_tr_vec, y_tr)
    score = f1_score(y_val, clf.predict(X_val_vec), average="macro")
    svc_scores[C] = score
    print(f"  C={C:>5}  val macro F1 = {score:.4f}")

best_C = max(svc_scores, key=svc_scores.get)
best_score = svc_scores[best_C]
print(f"\nЛучший C для SVC: {best_C} | val macro F1: {best_score:.4f}")


# Подбор C для LogisticRegression на val
print("\nПодбор C на val для LogisticRegression:")
lr_scores = {}
for C in [0.1, 0.3, 1.0, 3.0, 10.0]:
    clf = LogisticRegression(
        C=C, class_weight="balanced", solver="lbfgs",
        max_iter=2_000, tol=1e-4,
        random_state=RANDOM_STATE,
    )
    clf.fit(X_tr_vec, y_tr)
    score = f1_score(y_val, clf.predict(X_val_vec), average="macro")
    lr_scores[C] = score
    print(f"  C={C:>5}  val macro F1 = {score:.4f}")

best_C_lr = max(lr_scores, key=lr_scores.get)
best_score_lr = lr_scores[best_C_lr]
print(f"\nЛучший C для LogReg: {best_C_lr} | val macro F1: {best_score_lr:.4f}")


# Финальная модель + оценка на test
if best_score_lr > best_score:
    print(f"\nНа val побеждает LogisticRegression (C={best_C_lr}).")
    best_model = LogisticRegression(
        C=best_C_lr, class_weight="balanced", solver="lbfgs",
        max_iter=2_000, tol=1e-4,
        random_state=RANDOM_STATE,
    )
    best_model_name = "LogReg"
else:
    print(f"\nНа val побеждает LinearSVC (C={best_C}).")
    best_model = LinearSVC(
        C=best_C, class_weight="balanced", dual="auto",
        random_state=RANDOM_STATE, max_iter=15_000,
    )
    best_model_name = "LinearSVC"

best_model.fit(X_tr_vec, y_tr)
pred_svc = best_model.predict(X_test_vec)

print(f"\n=== {best_model_name} (best C) ===")
print("Accuracy:", round(accuracy_score(y_test, pred_svc), 4))
print("Macro F1:", round(f1_score(y_test, pred_svc, average="macro"), 4))
print("Weighted F1:", round(f1_score(y_test, pred_svc, average="weighted"), 4))
print()
print(classification_report(y_test, pred_svc, zero_division=0))
gc.collect()

Анализ ошибок на тестовой выборке

In [ ]:
import pandas as pd

err_df = pd.DataFrame({"text": X_test, "true": y_test, "pred": pred_svc})
errors = err_df[err_df["true"] != err_df["pred"]].copy()

print("Всего ошибок:", len(errors), f"({len(errors)/len(err_df)*100:.2f}%)")

confusions = errors.groupby(["true", "pred"]).size().sort_values(ascending=False)
print("\nТоп-20 смешений (истина -> предсказание):")
print(confusions.head(20))

for (true_label, pred_label), count in confusions.head(8).items():
    print(f"\n=== {true_label} -> {pred_label}: {count} ===")
    sample = errors[(errors["true"] == true_label) &
                    (errors["pred"] == pred_label)].head(3)
    for _, row in sample.iterrows():
        print("\nТекст:", " ".join(row["text"].split())[:400])
        print("Истина:", row["true"], "| Предсказано:", row["pred"])

Ансамбль

In [ ]:
import gc
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Обучаем обе модели на train
print("Обучаю LinearSVC...")
svc = LinearSVC(
    C=best_C, class_weight="balanced", dual="auto",
    random_state=RANDOM_STATE, max_iter=15_000,
)
svc.fit(X_tr_vec, y_tr)
gc.collect()

print("Обучаю LogisticRegression...")
lr = LogisticRegression(
    C=best_C_lr, class_weight="balanced", solver="lbfgs",
    max_iter=2_000, tol=1e-4,
    random_state=RANDOM_STATE,
)
lr.fit(X_tr_vec, y_tr)
gc.collect()

assert np.array_equal(svc.classes_, lr.classes_), "Порядок классов различается!"
classes = svc.classes_


def softmax(z, axis=1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


# Подбор весов на val
svc_val = softmax(svc.decision_function(X_val_vec))
lr_val = lr.predict_proba(X_val_vec)

print("\nПодбор весов ансамбля на val...")
best = (-1, None)
for w_svc in [0.0, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
    for w_lr in [0.0, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
        if w_svc == 0 and w_lr == 0:
            continue
        w = np.array([w_svc, w_lr]); w /= w.sum()
        blend = w[0] * svc_val + w[1] * lr_val
        pred = classes[blend.argmax(axis=1)]
        score = f1_score(y_val, pred, average="macro")
        if score > best[0]:
            best = (score, (w_svc, w_lr))

print(f"Лучшие веса [svc, lr]: {best[1]}")
print(f"Val macro F1 (ансамбль): {best[0]:.4f}")

svc_val_score = f1_score(y_val, classes[svc_val.argmax(axis=1)], average="macro")
lr_val_score = f1_score(y_val, classes[lr_val.argmax(axis=1)], average="macro")
print(f"Val macro F1 (только SVC): {svc_val_score:.4f}")
print(f"Val macro F1 (только LR):  {lr_val_score:.4f}")

# Выбор финального предиктора
use_ensemble = best[0] > max(svc_val_score, lr_val_score) + 1e-4
if not use_ensemble:
    print("\nАнсамбль не лучше лучшей одиночной модели — на test берём одиночную.")
    if lr_val_score >= svc_val_score:
        pred_final = lr.predict(X_test_vec)
        label = "LogReg (одиночная)"
    else:
        pred_final = svc.predict(X_test_vec)
        label = "LinearSVC (одиночная)"
else:
    print("\nАнсамбль побеждает на val — используем его на test.")
    svc_test = softmax(svc.decision_function(X_test_vec))
    lr_test = lr.predict_proba(X_test_vec)
    w = np.array(best[1]); w /= w.sum()
    blend = w[0] * svc_test + w[1] * lr_test
    pred_final = classes[blend.argmax(axis=1)]
    label = f"Ансамбль SVC+LR (w={tuple(best[1])})"

# Финальная оценка на test
print(f"\nФИНАЛ: {label}")
print("Accuracy:", round(accuracy_score(y_test, pred_final), 4))
print("Macro F1:", round(f1_score(y_test, pred_final, average="macro"), 4))
print("Weighted F1:", round(f1_score(y_test, pred_final, average="weighted"), 4))
print()
print(classification_report(y_test, pred_final, zero_division=0))
gc.collect()

## 1. Предобработка

- Отбор рубрик и маппинг 24 исходных в 4 целевые темы:
  - Развлечения = Культура, Ценности, Культпросвет
  - Спорт = Спорт
  - Технологии = Наука и техника, Интернет и СМИ
  - Новости = Россия, Мир, Бывший СССР, Экономика, Бизнес и др.
- Балансировка: до 30 000 записей на класс.
- Очистка: удаление HTML, URL, email; приведение к нижнему регистру; снятие пунктуации; обрезка до 1 200 символов.
- Лемматизация через pymorphy3 с кешем лемм (параллельно joblib.Parallel с threading).
- Удаление дубликатов по text_clean до сплита.
- Сплит: 80% train / 20% test, внутри train 85% / 15% (train/val).

## 2. Метод классификации

FeatureUnion(word TF-IDF 1–2 + char_wb TF-IDF 3–5) + LinearSVC / LogisticRegression.

Подбор C из набора {0.03, 0.1, 0.3, 1, 3, 10} — на отдельной валидационной выборке через f1_macro, без использования GridSearchCV на уже посчитанной матрице. Это устраняет утечку признаков: словарь и IDF обучаются только на X_tr

**Обоснование.** TF-IDF эффективен на разреженных текстовых данных и не требует GPU. Word + char n-grams покрывают лексику и морфологию без контекстных эмбеддингов. LinearSVC быстр, устойчив на многоклассовой задаче и стабильно даёт высокий результат. class_weight=balanced компенсирует дисбаланс. Сведение 24 рубрик в 4 темы устраняет главный источник ошибок — пересечение подкатегорий.

## 3. Результаты

| Модель | Accuracy | Macro F1 |
|---|---|---|
| LinearSVC (C=0.3) | 0.9399 | 0.9398 |
| LogisticRegression (C=3.0) | 0.9357 | 0.9361 |
| Ансамбль SVC+LR | не лучше одиночной | — |


## 4. Анализ ошибок

Ошибок: 1 441 из 23 986 (6.01%). Основные смешения:

- Новости / Технологии — 749 (52% всех ошибок): IT-компании, телеком, кибербезопасность в новостной повестке.
- Новости / Развлечения — 281: новости шоу-бизнеса и культурных событий с политическим контекстом.
- Технологии / Развлечения — 248: интернет-мемы, блогеры, стриминг.
- Новости / Спорт — 75: спортивная политика, допинговые скандалы.

Причины: пересечение тематик, короткие тексты без явных маркеров, дисбаланс сложности между классами.
Возможные улучшения: Использование поля tags как дополнительного признака(теги сильно коррелируют с рубрикой и в исходном датасете заполнены), расширение обучающей выборки.

## 5. Итог

Пайплайн классификации новостных текстов без трансформеров достигает точности ≥ 0.94 на отложенной тестовой выборке. Ключевые компоненты: лемматизация pymorphy3, свёртка 24 рубрик в 4 темы, word+char TF-IDF, подбор C на отдельной val-выборке, удаление дубликатов до сплита.